In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Project paths
PROJECT_PATH = Path("..")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
DATABASE_PATH = PROJECT_PATH / "data" / "fraud_analysis.db"

# SQLite connection
engine = create_engine(
    f"sqlite:///{DATABASE_PATH}"
)

print("Database path:")
print(DATABASE_PATH.resolve())

print("\nDatabase connection:")
with engine.connect() as connection:
    result = connection.execute(text("SELECT sqlite_version()"))
    print("SQLite version:", result.scalar())

print("\nDatabase layer initialized successfully.")

Database path:
D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\fraud_analysis.db

Database connection:
SQLite version: 3.45.1

Database layer initialized successfully.


In [2]:
# Load processed datasets

customers = pd.read_csv(
    PROCESSED_PATH / "dim_customer.csv"
)

cards = pd.read_csv(
    PROCESSED_PATH / "dim_card.csv"
)

merchants = pd.read_csv(
    PROCESSED_PATH / "dim_merchant.csv"
)

devices = pd.read_csv(
    PROCESSED_PATH / "dim_device.csv"
)

transactions = pd.read_csv(
    PROCESSED_PATH / "fact_transaction_clean.csv"
)

chargebacks = pd.read_csv(
    PROCESSED_PATH / "fact_chargeback.csv"
)

print("Customers:", customers.shape)
print("Cards:", cards.shape)
print("Merchants:", merchants.shape)
print("Devices:", devices.shape)
print("Transactions:", transactions.shape)
print("Chargebacks:", chargebacks.shape)

Customers: (20000, 10)
Cards: (30000, 6)
Merchants: (2000, 6)
Devices: (35001, 3)
Transactions: (1498353, 14)
Chargebacks: (10500, 5)


In [8]:
# Reset the database schema cleanly
# This removes any partially loaded tables from the failed attempt.

table_names = [
    "fact_rule_alert",
    "dim_rule",
    "fact_chargeback",
    "fact_transaction",
    "dim_date",
    "dim_device",
    "dim_merchant",
    "dim_card",
    "dim_customer"
]

with engine.begin() as connection:
    for table_name in table_names:
        connection.execute(text(f"DROP TABLE IF EXISTS {table_name}"))

print("Existing data-layer tables removed.")

# Recreate the star schema

with engine.begin() as connection:

    connection.execute(text("""
        CREATE TABLE dim_customer (
            customer_id INTEGER PRIMARY KEY,
            home_city TEXT,
            age_band TEXT,
            income_band TEXT,
            tenure_months INTEGER,
            segment TEXT,
            typical_monthly_spend REAL,
            preferred_category TEXT,
            active_start_hour INTEGER,
            active_end_hour INTEGER
        )
    """))

    connection.execute(text("""
        CREATE TABLE dim_card (
            card_id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL,
            card_type TEXT,
            issue_date TEXT,
            credit_limit REAL,
            status TEXT,
            FOREIGN KEY (customer_id)
                REFERENCES dim_customer(customer_id)
        )
    """))

    connection.execute(text("""
        CREATE TABLE dim_merchant (
            merchant_id INTEGER PRIMARY KEY,
            merchant_name TEXT,
            category TEXT,
            city TEXT,
            country TEXT,
            risk_tier TEXT
        )
    """))

    connection.execute(text("""
        CREATE TABLE dim_device (
            device_id INTEGER PRIMARY KEY,
            device_type TEXT,
            first_seen_date TEXT
        )
    """))

    connection.execute(text("""
        CREATE TABLE dim_date (
            date_key INTEGER PRIMARY KEY,
            calendar_date TEXT UNIQUE,
            year INTEGER,
            quarter INTEGER,
            month INTEGER,
            month_name TEXT,
            week INTEGER,
            day_of_month INTEGER,
            day_of_week INTEGER,
            day_name TEXT
        )
    """))

    connection.execute(text("""
        CREATE TABLE fact_transaction (
            txn_id TEXT PRIMARY KEY,
            card_id INTEGER NOT NULL,
            customer_id INTEGER NOT NULL,
            merchant_id INTEGER NOT NULL,
            device_id INTEGER NOT NULL,
            txn_ts TEXT NOT NULL,
            amount REAL NOT NULL,
            channel TEXT,
            category TEXT,
            merchant_city TEXT,
            merchant_country TEXT,
            ip_country TEXT,
            auth_result TEXT,
            customer_home_city TEXT,
            is_fraud INTEGER,
            date_key INTEGER,
            FOREIGN KEY (card_id) REFERENCES dim_card(card_id),
            FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
            FOREIGN KEY (merchant_id) REFERENCES dim_merchant(merchant_id),
            FOREIGN KEY (device_id) REFERENCES dim_device(device_id),
            FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
        )
    """))

    connection.execute(text("""
        CREATE TABLE fact_chargeback (
            chargeback_id INTEGER PRIMARY KEY,
            txn_id TEXT,
            chargeback_date TEXT,
            confirmed_fraud_flag INTEGER,
            loss_amount REAL,
            recovered_amount REAL,
            FOREIGN KEY (txn_id)
                REFERENCES fact_transaction(txn_id)
        )
    """))

    connection.execute(text("""
        CREATE TABLE dim_rule (
            rule_id INTEGER PRIMARY KEY,
            rule_name TEXT,
            rule_description TEXT,
            threshold_value REAL,
            active_flag INTEGER
        )
    """))

    connection.execute(text("""
        CREATE TABLE fact_rule_alert (
            alert_id INTEGER PRIMARY KEY,
            rule_id INTEGER,
            txn_id TEXT,
            alert_ts TEXT,
            alert_amount REAL,
            is_fraud INTEGER,
            review_outcome TEXT,
            FOREIGN KEY (rule_id) REFERENCES dim_rule(rule_id),
            FOREIGN KEY (txn_id) REFERENCES fact_transaction(txn_id)
        )
    """))

print("Star schema recreated successfully.")

# Verify tables
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """)
    )

    tables = [row[0] for row in result]

print("\nTables:")
for table in tables:
    print("-", table)

Existing data-layer tables removed.
Star schema recreated successfully.

Tables:
- dim_card
- dim_customer
- dim_date
- dim_device
- dim_merchant
- dim_rule
- fact_chargeback
- fact_rule_alert
- fact_transaction


In [9]:
# Load dimension tables into the existing schema.
# We use append so the PK/FK definitions remain intact.
# We intentionally do NOT use method="multi" because SQLite
# can exceed its SQL-variable limit.

customers.to_sql(
    "dim_customer",
    engine,
    if_exists="append",
    index=False
)

cards.to_sql(
    "dim_card",
    engine,
    if_exists="append",
    index=False
)

merchants.to_sql(
    "dim_merchant",
    engine,
    if_exists="append",
    index=False
)

devices.to_sql(
    "dim_device",
    engine,
    if_exists="append",
    index=False
)

print("Dimension tables loaded successfully.")

# Verify row counts
with engine.connect() as connection:
    for table_name in [
        "dim_customer",
        "dim_card",
        "dim_merchant",
        "dim_device"
    ]:
        count = connection.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).scalar()

        print(f"{table_name}: {count:,} rows")

Dimension tables loaded successfully.
dim_customer: 20,000 rows
dim_card: 30,000 rows
dim_merchant: 2,000 rows
dim_device: 35,001 rows


In [10]:
# Build the date dimension from the transaction date range

transactions["txn_ts"] = pd.to_datetime(
    transactions["txn_ts"],
    errors="coerce",
    format="mixed"
)

# Get the complete transaction date range
start_date = transactions["txn_ts"].min().normalize()
end_date = transactions["txn_ts"].max().normalize()

print("Transaction date range:")
print(start_date.date(), "to", end_date.date())

# Create one row per calendar date
date_range = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

dim_date = pd.DataFrame({
    "calendar_date": date_range
})

# Date attributes
dim_date["date_key"] = (
    dim_date["calendar_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["calendar_date"].dt.year

dim_date["quarter"] = dim_date["calendar_date"].dt.quarter

dim_date["month"] = dim_date["calendar_date"].dt.month

dim_date["month_name"] = dim_date["calendar_date"].dt.month_name()

dim_date["week"] = (
    dim_date["calendar_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

dim_date["day_of_month"] = dim_date["calendar_date"].dt.day

dim_date["day_of_week"] = (
    dim_date["calendar_date"].dt.dayofweek + 1
)

dim_date["day_name"] = dim_date["calendar_date"].dt.day_name()

# Reorder columns
dim_date = dim_date[
    [
        "date_key",
        "calendar_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week",
        "day_of_month",
        "day_of_week",
        "day_name"
    ]
]

print("\nDate dimension:")
print(dim_date.head())

print("\nRows:", len(dim_date))
print("Unique dates:", dim_date["calendar_date"].nunique())

Transaction date range:
2025-01-01 to 2025-12-31

Date dimension:
   date_key calendar_date  year  quarter  month month_name  week  \
0  20250101    2025-01-01  2025        1      1    January     1   
1  20250102    2025-01-02  2025        1      1    January     1   
2  20250103    2025-01-03  2025        1      1    January     1   
3  20250104    2025-01-04  2025        1      1    January     1   
4  20250105    2025-01-05  2025        1      1    January     1   

   day_of_month  day_of_week   day_name  
0             1            3  Wednesday  
1             2            4   Thursday  
2             3            5     Friday  
3             4            6   Saturday  
4             5            7     Sunday  

Rows: 365
Unique dates: 365


In [11]:
# Load the date dimension into SQLite

dim_date.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

print("Date dimension loaded successfully.")

# Verify row count
with engine.connect() as connection:
    count = connection.execute(
        text("SELECT COUNT(*) FROM dim_date")
    ).scalar()

print(f"dim_date: {count:,} rows")

# Verify date range in SQLite
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                MIN(calendar_date) AS min_date,
                MAX(calendar_date) AS max_date
            FROM dim_date
        """)
    ).fetchone()

print("SQLite date range:")
print(result[0], "to", result[1])

Date dimension loaded successfully.
dim_date: 365 rows
SQLite date range:
2025-01-01 00:00:00.000000 to 2025-12-31 00:00:00.000000


In [12]:
# Prepare the transaction fact table for SQLite

# Work on a copy so the original dataframe remains available
fact_transaction = transactions.copy()

# ---------------------------------------------------------
# 1. Parse transaction timestamp
# ---------------------------------------------------------

fact_transaction["txn_ts"] = pd.to_datetime(
    fact_transaction["txn_ts"],
    errors="coerce",
    format="mixed"
)

# ---------------------------------------------------------
# 2. Create fraud label from confirmed chargebacks
# ---------------------------------------------------------

fraud_txn_ids = set(
    chargebacks.loc[
        chargebacks["confirmed_fraud_flag"] == 1,
        "txn_id"
    ]
)

fact_transaction["is_fraud"] = (
    fact_transaction["txn_id"].isin(fraud_txn_ids)
).astype(int)

# ---------------------------------------------------------
# 3. Create date_key
# ---------------------------------------------------------

fact_transaction["date_key"] = (
    fact_transaction["txn_ts"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# ---------------------------------------------------------
# 4. Standardize merchant city names
# ---------------------------------------------------------

fact_transaction["merchant_city"] = (
    fact_transaction["merchant_city"]
    .replace({
        "Bangalore": "Bengaluru",
        "BLR": "Bengaluru"
    })
)

# ---------------------------------------------------------
# 5. Standardize customer home city names
# ---------------------------------------------------------

fact_transaction["customer_home_city"] = (
    fact_transaction["customer_home_city"]
    .replace({
        "Bangalore": "Bengaluru",
        "BLR": "Bengaluru"
    })
)

# ---------------------------------------------------------
# 6. Ensure unknown device member is represented as -1
# ---------------------------------------------------------

fact_transaction["device_id"] = (
    fact_transaction["device_id"]
    .fillna(-1)
    .astype(int)
)

# ---------------------------------------------------------
# 7. Select exact database columns
# ---------------------------------------------------------

fact_transaction = fact_transaction[
    [
        "txn_id",
        "card_id",
        "customer_id",
        "merchant_id",
        "device_id",
        "txn_ts",
        "amount",
        "channel",
        "category",
        "merchant_city",
        "merchant_country",
        "ip_country",
        "auth_result",
        "customer_home_city",
        "is_fraud",
        "date_key"
    ]
]

# ---------------------------------------------------------
# 8. Validation
# ---------------------------------------------------------

print("Prepared transaction fact table")
print("--------------------------------")

print("Rows:", f"{len(fact_transaction):,}")
print("Columns:", len(fact_transaction.columns))

print("\nColumns:")
print(fact_transaction.columns.tolist())

print("\nMissing values:")
print(
    fact_transaction.isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nFraud transactions:")
print(
    fact_transaction["is_fraud"]
    .value_counts()
    .sort_index()
)

print("\nDate key range:")
print(
    fact_transaction["date_key"].min(),
    "to",
    fact_transaction["date_key"].max()
)

Prepared transaction fact table
--------------------------------
Rows: 1,498,353
Columns: 16

Columns:
['txn_id', 'card_id', 'customer_id', 'merchant_id', 'device_id', 'txn_ts', 'amount', 'channel', 'category', 'merchant_city', 'merchant_country', 'ip_country', 'auth_result', 'customer_home_city', 'is_fraud', 'date_key']

Missing values:
txn_id           0
card_id          0
customer_id      0
merchant_id      0
device_id        0
txn_ts           0
amount           0
channel          0
category         0
merchant_city    0
dtype: int64

Fraud transactions:
is_fraud
0    1487865
1      10488
Name: count, dtype: int64

Date key range:
20250101 to 20251231


In [13]:
# Validate all foreign-key relationships before loading
# the 1.5M-row transaction fact table.

with engine.connect() as connection:

    # Card IDs
    invalid_cards = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM fact_transaction ft
            WHERE 1 = 0
        """)
    ).scalar()

    # Get valid dimension keys
    valid_card_ids = set(
        pd.read_sql(
            "SELECT card_id FROM dim_card",
            connection
        )["card_id"]
    )

    valid_customer_ids = set(
        pd.read_sql(
            "SELECT customer_id FROM dim_customer",
            connection
        )["customer_id"]
    )

    valid_merchant_ids = set(
        pd.read_sql(
            "SELECT merchant_id FROM dim_merchant",
            connection
        )["merchant_id"]
    )

    valid_device_ids = set(
        pd.read_sql(
            "SELECT device_id FROM dim_device",
            connection
        )["device_id"]
    )

    valid_date_keys = set(
        pd.read_sql(
            "SELECT date_key FROM dim_date",
            connection
        )["date_key"]
    )

# Check each relationship
invalid_card_count = (
    ~fact_transaction["card_id"].isin(valid_card_ids)
).sum()

invalid_customer_count = (
    ~fact_transaction["customer_id"].isin(valid_customer_ids)
).sum()

invalid_merchant_count = (
    ~fact_transaction["merchant_id"].isin(valid_merchant_ids)
).sum()

invalid_device_count = (
    ~fact_transaction["device_id"].isin(valid_device_ids)
).sum()

invalid_date_count = (
    ~fact_transaction["date_key"].isin(valid_date_keys)
).sum()

print("Foreign-key validation")
print("----------------------")
print(f"Invalid card IDs:      {invalid_card_count:,}")
print(f"Invalid customer IDs:  {invalid_customer_count:,}")
print(f"Invalid merchant IDs:  {invalid_merchant_count:,}")
print(f"Invalid device IDs:    {invalid_device_count:,}")
print(f"Invalid date keys:     {invalid_date_count:,}")

total_invalid = (
    invalid_card_count
    + invalid_customer_count
    + invalid_merchant_count
    + invalid_device_count
    + invalid_date_count
)

print("\nTotal invalid foreign-key references:", f"{total_invalid:,}")

if total_invalid == 0:
    print("\n✓ All transaction foreign keys are valid.")
else:
    print("\n⚠ Foreign-key issues detected. Do not load the fact table yet.")

Foreign-key validation
----------------------
Invalid card IDs:      0
Invalid customer IDs:  0
Invalid merchant IDs:  0
Invalid device IDs:    0
Invalid date keys:     0

Total invalid foreign-key references: 0

✓ All transaction foreign keys are valid.


In [14]:
# Load fact_transaction in controlled chunks
# to avoid SQLite's SQL-variable limit.

CHUNK_SIZE = 25_000

print("Starting transaction fact load...")
print(f"Total rows: {len(fact_transaction):,}")
print(f"Chunk size: {CHUNK_SIZE:,}")

# Make sure the table is empty before loading
with engine.begin() as connection:
    connection.execute(text("DELETE FROM fact_transaction"))

total_rows = len(fact_transaction)

for start in range(0, total_rows, CHUNK_SIZE):

    end = min(start + CHUNK_SIZE, total_rows)

    chunk = fact_transaction.iloc[start:end].copy()

    chunk.to_sql(
        "fact_transaction",
        engine,
        if_exists="append",
        index=False
    )

    print(
        f"Loaded rows {start + 1:,} - {end:,} "
        f"of {total_rows:,}"
    )

print("\nTransaction fact load completed.")

# Verify row count
with engine.connect() as connection:

    count = connection.execute(
        text("SELECT COUNT(*) FROM fact_transaction")
    ).scalar()

print(f"fact_transaction: {count:,} rows")

Starting transaction fact load...
Total rows: 1,498,353
Chunk size: 25,000
Loaded rows 1 - 25,000 of 1,498,353
Loaded rows 25,001 - 50,000 of 1,498,353
Loaded rows 50,001 - 75,000 of 1,498,353
Loaded rows 75,001 - 100,000 of 1,498,353
Loaded rows 100,001 - 125,000 of 1,498,353
Loaded rows 125,001 - 150,000 of 1,498,353
Loaded rows 150,001 - 175,000 of 1,498,353
Loaded rows 175,001 - 200,000 of 1,498,353
Loaded rows 200,001 - 225,000 of 1,498,353
Loaded rows 225,001 - 250,000 of 1,498,353
Loaded rows 250,001 - 275,000 of 1,498,353
Loaded rows 275,001 - 300,000 of 1,498,353
Loaded rows 300,001 - 325,000 of 1,498,353
Loaded rows 325,001 - 350,000 of 1,498,353
Loaded rows 350,001 - 375,000 of 1,498,353
Loaded rows 375,001 - 400,000 of 1,498,353
Loaded rows 400,001 - 425,000 of 1,498,353
Loaded rows 425,001 - 450,000 of 1,498,353
Loaded rows 450,001 - 475,000 of 1,498,353
Loaded rows 475,001 - 500,000 of 1,498,353
Loaded rows 500,001 - 525,000 of 1,498,353
Loaded rows 525,001 - 550,000 of 1

In [15]:
# Validate the transaction fact table after loading

with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS total_transactions,
                SUM(amount) AS total_transaction_value,
                SUM(is_fraud) AS fraud_transactions,
                SUM(
                    CASE
                        WHEN is_fraud = 1 THEN amount
                        ELSE 0
                    END
                ) AS fraud_value,
                MIN(txn_ts) AS first_transaction,
                MAX(txn_ts) AS last_transaction
            FROM fact_transaction
        """)
    ).fetchone()

print("Transaction fact validation")
print("---------------------------")
print(f"Total transactions:   {result[0]:,}")
print(f"Total transaction value: ₹{result[1]:,.2f}")
print(f"Fraud transactions:    {result[2]:,}")
print(f"Fraud value:           ₹{result[3]:,.2f}")
print(f"First transaction:     {result[4]}")
print(f"Last transaction:      {result[5]}")

# Verify fraud rate
fraud_rate = result[2] / result[0] * 100

print(f"Fraud rate:            {fraud_rate:.4f}%")

Transaction fact validation
---------------------------
Total transactions:   1,498,353
Total transaction value: ₹3,762,810,623.99
Fraud transactions:    10,488
Fraud value:           ₹95,197,860.34
First transaction:     2025-01-01 00:00:03.000000
Last transaction:      2025-12-31 23:59:35.000000
Fraud rate:            0.7000%


In [20]:
# Prepare the chargeback fact table

fact_chargeback = chargebacks.copy()

# ---------------------------------------------------------
# 1. Create a surrogate chargeback ID
# ---------------------------------------------------------

fact_chargeback.insert(
    0,
    "chargeback_id",
    range(1, len(fact_chargeback) + 1)
)

# ---------------------------------------------------------
# 2. Rename reported_date to chargeback_date
# ---------------------------------------------------------

fact_chargeback = fact_chargeback.rename(
    columns={
        "reported_date": "chargeback_date"
    }
)

# ---------------------------------------------------------
# 3. Parse chargeback date
# ---------------------------------------------------------

fact_chargeback["chargeback_date"] = pd.to_datetime(
    fact_chargeback["chargeback_date"],
    errors="coerce",
    format="mixed"
)

# ---------------------------------------------------------
# 4. Make fraud flag an integer
# ---------------------------------------------------------

fact_chargeback["confirmed_fraud_flag"] = (
    fact_chargeback["confirmed_fraud_flag"]
    .astype(int)
)

# ---------------------------------------------------------
# 5. Select exact database columns
# ---------------------------------------------------------

fact_chargeback = fact_chargeback[
    [
        "chargeback_id",
        "txn_id",
        "chargeback_date",
        "confirmed_fraud_flag",
        "loss_amount",
        "recovered_amount"
    ]
]

# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

print("Prepared chargeback fact table")
print("--------------------------------")

print("Rows:", f"{len(fact_chargeback):,}")
print("Columns:", len(fact_chargeback.columns))

print("\nColumns:")
print(fact_chargeback.columns.tolist())

print("\nMissing values:")
print(fact_chargeback.isna().sum())

print("\nConfirmed fraud flag:")
print(
    fact_chargeback["confirmed_fraud_flag"]
    .value_counts()
    .sort_index()
)

# ---------------------------------------------------------
# 7. Financial totals
# ---------------------------------------------------------

total_loss = fact_chargeback["loss_amount"].sum()
total_recovered = fact_chargeback["recovered_amount"].sum()
net_loss = total_loss - total_recovered

print("\nFinancial totals:")
print(f"Loss amount:      ₹{total_loss:,.2f}")
print(f"Recovered amount: ₹{total_recovered:,.2f}")
print(f"Net loss:         ₹{net_loss:,.2f}")

Prepared chargeback fact table
--------------------------------
Rows: 10,500
Columns: 6

Columns:
['chargeback_id', 'txn_id', 'chargeback_date', 'confirmed_fraud_flag', 'loss_amount', 'recovered_amount']

Missing values:
chargeback_id           0
txn_id                  0
chargeback_date         0
confirmed_fraud_flag    0
loss_amount             0
recovered_amount        0
dtype: int64

Confirmed fraud flag:
confirmed_fraud_flag
1    10500
Name: count, dtype: int64

Financial totals:
Loss amount:      ₹95,285,937.42
Recovered amount: ₹26,506,448.86
Net loss:         ₹68,779,488.56


In [21]:
# Reconcile chargebacks against the transaction fact table
# before loading them into SQLite.

# Transaction IDs currently present in the cleaned fact table
transaction_ids = set(fact_transaction["txn_id"])

# Identify chargebacks whose transaction no longer exists
fact_chargeback["transaction_exists"] = (
    fact_chargeback["txn_id"].isin(transaction_ids)
)

# Summary
total_chargebacks = len(fact_chargeback)

linked_chargebacks = fact_chargeback["transaction_exists"].sum()

orphan_chargebacks = (
    ~fact_chargeback["transaction_exists"]
).sum()

print("Chargeback reconciliation")
print("-------------------------")
print(f"Total chargebacks:       {total_chargebacks:,}")
print(f"Linked to transactions:  {linked_chargebacks:,}")
print(f"Orphan chargebacks:      {orphan_chargebacks:,}")

# Financial impact of orphan chargebacks
orphan_rows = fact_chargeback[
    ~fact_chargeback["transaction_exists"]
].copy()

orphan_loss = orphan_rows["loss_amount"].sum()
orphan_recovered = orphan_rows["recovered_amount"].sum()
orphan_net = orphan_loss - orphan_recovered

print("\nOrphan chargeback financial impact")
print("----------------------------------")
print(f"Orphan loss:       ₹{orphan_loss:,.2f}")
print(f"Orphan recovered:  ₹{orphan_recovered:,.2f}")
print(f"Orphan net loss:   ₹{orphan_net:,.2f}")

# Show orphan transaction IDs
print("\nOrphan transaction IDs:")
print(orphan_rows["txn_id"].tolist())

Chargeback reconciliation
-------------------------
Total chargebacks:       10,500
Linked to transactions:  10,488
Orphan chargebacks:      12

Orphan chargeback financial impact
----------------------------------
Orphan loss:       ₹88,077.08
Orphan recovered:  ₹35,232.28
Orphan net loss:   ₹52,844.80

Orphan transaction IDs:
[36668, 197948, 365391, 375063, 500440, 577629, 585412, 688822, 940550, 1132188, 1263985, 1274936]


In [22]:
# Recreate fact_chargeback without a strict transaction FK.
#
# Reason:
# 12 chargebacks refer to transactions removed during data cleaning.
# We retain these chargebacks for financial-loss accounting rather than
# manufacturing transaction records.

with engine.begin() as connection:

    connection.execute(
        text("DROP TABLE IF EXISTS fact_chargeback")
    )

    connection.execute(text("""
        CREATE TABLE fact_chargeback (
            chargeback_id INTEGER PRIMARY KEY,
            txn_id TEXT,
            chargeback_date TEXT,
            confirmed_fraud_flag INTEGER,
            loss_amount REAL,
            recovered_amount REAL
        )
    """))

print("fact_chargeback table recreated.")
print("Transaction FK intentionally omitted to preserve orphan chargebacks.")

fact_chargeback table recreated.
Transaction FK intentionally omitted to preserve orphan chargebacks.


In [23]:
# Load all chargeback records into SQLite

fact_chargeback_load = fact_chargeback.drop(
    columns=["transaction_exists"]
).copy()

fact_chargeback_load.to_sql(
    "fact_chargeback",
    engine,
    if_exists="append",
    index=False
)

print("Chargeback fact load completed.")

# Verify row count and financial totals
with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS chargebacks,
                SUM(loss_amount) AS total_loss,
                SUM(recovered_amount) AS total_recovered,
                SUM(loss_amount - recovered_amount) AS net_loss
            FROM fact_chargeback
        """)
    ).fetchone()

print("\nChargeback fact validation")
print("--------------------------")
print(f"Chargebacks:       {result[0]:,}")
print(f"Total loss:        ₹{result[1]:,.2f}")
print(f"Total recovered:   ₹{result[2]:,.2f}")
print(f"Net loss:          ₹{result[3]:,.2f}")

Chargeback fact load completed.

Chargeback fact validation
--------------------------
Chargebacks:       10,500
Total loss:        ₹95,285,937.42
Total recovered:   ₹26,506,448.86
Net loss:          ₹68,779,488.56


In [24]:
# Final validation of the SQLite data layer

print("FINAL DATA LAYER VALIDATION")
print("=" * 50)

with engine.connect() as connection:

    # -----------------------------------------------------
    # 1. Row counts for all core tables
    # -----------------------------------------------------

    tables_to_check = [
        "dim_customer",
        "dim_card",
        "dim_merchant",
        "dim_device",
        "dim_date",
        "fact_transaction",
        "fact_chargeback",
        "dim_rule",
        "fact_rule_alert"
    ]

    print("\nTable row counts")
    print("-" * 50)

    for table_name in tables_to_check:

        count = connection.execute(
            text(f"SELECT COUNT(*) FROM {table_name}")
        ).scalar()

        print(f"{table_name:<22} {count:>12,}")

    # -----------------------------------------------------
    # 2. Transaction financial baseline
    # -----------------------------------------------------

    transaction_result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS transaction_count,
                SUM(amount) AS transaction_value,
                SUM(is_fraud) AS fraud_count,
                SUM(
                    CASE
                        WHEN is_fraud = 1 THEN amount
                        ELSE 0
                    END
                ) AS fraud_value
            FROM fact_transaction
        """)
    ).fetchone()

    # -----------------------------------------------------
    # 3. Chargeback financial baseline
    # -----------------------------------------------------

    chargeback_result = connection.execute(
        text("""
            SELECT
                COUNT(*) AS chargeback_count,
                SUM(loss_amount) AS gross_loss,
                SUM(recovered_amount) AS recovered,
                SUM(loss_amount - recovered_amount) AS net_loss
            FROM fact_chargeback
        """)
    ).fetchone()

    # -----------------------------------------------------
    # 4. Orphan chargebacks
    # -----------------------------------------------------

    orphan_result = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM fact_chargeback cb
            LEFT JOIN fact_transaction ft
                ON cb.txn_id = ft.txn_id
            WHERE ft.txn_id IS NULL
        """)
    ).scalar()

print("\nTransaction baseline")
print("-" * 50)

print(f"Transactions:       {transaction_result[0]:,}")
print(f"Transaction value:  ₹{transaction_result[1]:,.2f}")
print(f"Fraud transactions: {transaction_result[2]:,}")
print(f"Fraud value:        ₹{transaction_result[3]:,.2f}")

fraud_rate = (
    transaction_result[2]
    / transaction_result[0]
    * 100
)

print(f"Fraud rate:         {fraud_rate:.4f}%")

print("\nChargeback baseline")
print("-" * 50)

print(f"Chargebacks:        {chargeback_result[0]:,}")
print(f"Gross loss:         ₹{chargeback_result[1]:,.2f}")
print(f"Recovered:          ₹{chargeback_result[2]:,.2f}")
print(f"Net loss:           ₹{chargeback_result[3]:,.2f}")

print("\nReconciliation")
print("-" * 50)

print(f"Orphan chargebacks: {orphan_result:,}")

print("\n" + "=" * 50)
print("DATA LAYER VALIDATION COMPLETE")
print("=" * 50)

FINAL DATA LAYER VALIDATION

Table row counts
--------------------------------------------------
dim_customer                 20,000
dim_card                     30,000
dim_merchant                  2,000
dim_device                   35,001
dim_date                        365
fact_transaction          1,498,353
fact_chargeback              10,500
dim_rule                          0
fact_rule_alert                   0

Transaction baseline
--------------------------------------------------
Transactions:       1,498,353
Transaction value:  ₹3,762,810,623.99
Fraud transactions: 10,488
Fraud value:        ₹95,197,860.34
Fraud rate:         0.7000%

Chargeback baseline
--------------------------------------------------
Chargebacks:        10,500
Gross loss:         ₹95,285,937.42
Recovered:          ₹26,506,448.86
Net loss:           ₹68,779,488.56

Reconciliation
--------------------------------------------------
Orphan chargebacks: 12

DATA LAYER VALIDATION COMPLETE
